<p align="center">
  <a href="https://github.com/wavekat/wavekat-lab">
    <img src="https://github.com/wavekat/wavekat-brand/raw/main/assets/banners/wavekat-lab-narrow.svg" alt="WaveKat Lab">
  </a>
</p>

# 05 — Score candidates with an LLM

Send each candidate's transcript to an LLM via [OpenRouter](https://openrouter.ai)
and record an independent `END_OF_TURN` / `CONTINUATION` verdict.
This is the third consensus signal — alongside the structural label
(notebook 03) and the current model's verdict (notebook 04). Notebook
06 fuses the three.

**Backend.** OpenRouter exposes an OpenAI-compatible HTTP API across
many providers, so we can swap models without rewriting the notebook.

**Model choice — read this before you swap.** OpenRouter passes through
to upstream providers, each with its own Terms of Service. Some closed
models (Claude, GPT) prohibit using outputs to train other models.
**The verdicts produced here become labels in our training set, so the
chosen model's ToS must allow that.** The defaults below
(`deepseek-chat`, Qwen open-weights) are picked specifically because
their providers permit using outputs as training data. If you swap in
another model, re-check its ToS first.

**Independence from the structural signal.** We feed the LLM only
`text` (the actual clip transcript) — *not* `parent_text` or
`gap_to_next_s`. Those leak the structural label (`speaker_change` rows
have `text == parent_text`; `intra_utterance_cut` rows don't), so
including them would collapse this into a copy of notebook 03.

**Resumable.** Verdicts are appended line-by-line to
`llm_verdicts.jsonl`. Re-running the notebook skips already-scored
rows so a mid-run failure doesn't re-bill the API.

## Configure paths

In [55]:
from pathlib import Path

MINING_ROOT = Path("../../datasets/smart-turn-zh-mining").resolve()
SCORED_IN   = MINING_ROOT / "candidates_scored.parquet"
LLM_JSONL   = MINING_ROOT / "llm_verdicts.jsonl"
LLM_PARQUET = MINING_ROOT / "candidates_llm.parquet"
WAV_DIR     = Path("../../datasets/MagicData-RAMC/MDT2021S003/WAV").resolve()

print(f"scored in : {SCORED_IN.name}")
print(f"jsonl log : {LLM_JSONL.name}")
print(f"parquet   : {LLM_PARQUET.name}")
print(f"wavs      : {WAV_DIR.name}")
print("✅ paths configured")

scored in : candidates_scored.parquet
jsonl log : llm_verdicts.jsonl
parquet   : candidates_llm.parquet
wavs      : WAV
✅ paths configured


## OpenRouter config

`MODEL` is the OpenRouter model slug. The default is a **natively
non-reasoning** model — each call is fast and the response *is* the
verdict, not chain-of-thought.

- `moonshotai/kimi-k2-0905` — Moonshot Kimi K2 (2025-09-05). Strong
  Chinese capability. **Permits using outputs as training data** —
  Moonshot's terms allow synthetic data generation. ~1.4s/call,
  $0.40/M input + $2.00/M output → ~$4–5 for the full 34K run.

Why not the reasoning Kimi variants? `kimi-k2.5` / `kimi-k2.6` /
`kimi-k2-thinking` always spend tokens on hidden chain-of-thought
even when reasoning is suppressed via OpenRouter's
`reasoning.enabled=False` flag. With a small `max_tokens` budget the
verdict gets truncated; with a large budget you pay for thinking that
doesn't help this task. Empirical canary on `max_tokens=16`:
**4/4 for `kimi-k2-0905`** vs 1/4 for k2.5/k2.6.

Other training-data-friendly fallbacks: `deepseek/deepseek-chat`
(MIT-licensed weights, no thinking), `qwen/qwen-2.5-72b-instruct`
(Qwen license permits synthetic data). **The chosen model's ToS
must permit using outputs as training data**, since these verdicts
become labels in our dataset. Closed models (Claude, GPT) generally
do not — don't swap one in.

`CONCURRENCY` caps in-flight requests. Paid Kimi has generous
per-account rate limits, so 16 is comfortable. `MAX_TOKENS = 16` is
enough for one verdict word — hard cap keeps cost predictable.

**API key.** The cell below reads `OPENROUTER_API_KEY` from the
environment. To set it, copy `.env.example` at the repo root to `.env`
and fill in your key:

```bash
cp .env.example .env
# then edit .env and paste your real key
```

`.env` is already gitignored. The cell auto-discovers it by walking up
from the notebook directory and loads it via `python-dotenv`. As a
fallback you can `export OPENROUTER_API_KEY=...` in the shell that
launches Jupyter — anything already in the process environment wins
over `.env` (so a shell export takes precedence). The key is never
echoed, only a short prefix.

In [56]:
import os

MODEL             = "moonshotai/kimi-k2-0905"  # natively non-reasoning
CONCURRENCY       = 16
MAX_TOKENS        = 16         # one verdict word; hard cap on cost
TEMPERATURE       = 0.0
REASONING_EFFORT  = None       # "low" | "medium" | "high" | None (None = no reasoning)
REQUEST_TIMEOUT_S = 60
MAX_RETRIES       = 5
RATELIMIT_BACKOFF_S = 30       # default sleep on 429 if no Retry-After header

# Load OPENROUTER_API_KEY from a `.env` walking up from this notebook.
# `override=False` means an already-exported shell value wins over `.env`.
try:
    from dotenv import find_dotenv, load_dotenv
    dotenv_path = find_dotenv(usecwd=True)
    if dotenv_path:
        load_dotenv(dotenv_path, override=False)
        print(f"loaded .env  : {Path(dotenv_path).name} (from {Path(dotenv_path).parent.name}/)")
    else:
        print("loaded .env  : (none found — relying on shell environment)")
except ImportError:
    print("loaded .env  : (python-dotenv not installed — relying on shell environment)")

API_KEY = os.environ.get("OPENROUTER_API_KEY")
if not API_KEY:
    raise RuntimeError(
        "OPENROUTER_API_KEY not set.\n"
        "  1. Get a key at https://openrouter.ai/keys\n"
        "  2. cp .env.example .env  (at the repo root)\n"
        "  3. Edit .env and paste your key, then restart the kernel."
    )

# OpenRouter recommends sending these so usage shows up tagged.
OR_REFERER = "https://github.com/wavekat/wavekat-lab"
OR_TITLE   = "wavekat-lab smart-turn-zh mining"

print(f"model        : {MODEL}")
print(f"concurrency  : {CONCURRENCY}")
print(f"max tokens   : {MAX_TOKENS}")
print(f"reasoning    : {REASONING_EFFORT}")
print(f"temperature  : {TEMPERATURE}")
print(f"max retries  : {MAX_RETRIES}")
print(f"api key      : {API_KEY[:6]}…  (length {len(API_KEY)})")
print("✅ openrouter configured")

loaded .env  : .env (from wavekat-lab/)
model        : moonshotai/kimi-k2-0905
concurrency  : 16
max tokens   : 16
reasoning    : None
temperature  : 0.0
max retries  : 5
api key      : sk-or-…  (length 73)
✅ openrouter configured


## Load candidates and identify pending rows

We read `candidates_scored.parquet`, then skim `llm_verdicts.jsonl`
(if it exists) for `(session_id, candidate_idx)` keys that already
have a usable verdict (`END_OF_TURN` or `CONTINUATION`). Anything
else — `ERROR`, `UNKNOWN`, or absent — counts as **pending** so a
rerun retries it. The latest record for each key wins (so a
successful retry overrides a stale `ERROR`).

This is the resume-from-stop entry point: ctrl-C the scoring loop at
any time, re-run this cell + the loop, and only the unfinished work
re-fires. No re-billing on already-done rows.

In [57]:
import json
import pandas as pd

scored = pd.read_parquet(SCORED_IN)
print(f"candidates : {len(scored):,}")

USABLE = {"END_OF_TURN", "CONTINUATION"}

# Walk the JSONL once; latest record per key wins, then keep only the keys
# whose final verdict is usable. ERROR / UNKNOWN keys fall through to pending.
latest: dict[tuple[str, int], str] = {}
errors = 0
unknown = 0
if LLM_JSONL.exists():
    with LLM_JSONL.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                rec = json.loads(line)
            except json.JSONDecodeError:
                continue
            latest[(rec["session_id"], int(rec["candidate_idx"]))] = rec.get("verdict", "")

done_keys = {k for k, v in latest.items() if v in USABLE}
errors  = sum(1 for v in latest.values() if v == "ERROR")
unknown = sum(1 for v in latest.values() if v == "UNKNOWN")

print(f"jsonl records  : {len(latest):,} (latest per key)")
print(f"  usable       : {len(done_keys):,}  (ERROR + UNKNOWN will be retried)")
print(f"  ERROR        : {errors:,}")
print(f"  UNKNOWN      : {unknown:,}")

key_tuples = list(zip(scored["session_id"].tolist(), scored["candidate_idx"].tolist()))
pending_mask = [k not in done_keys for k in key_tuples]
pending = scored.loc[pending_mask].reset_index(drop=True)
print(f"pending        : {len(pending):,} / {len(scored):,}")
print("✅ pending workload computed")

candidates : 33,943
jsonl records  : 0 (latest per key)
  usable       : 0  (ERROR + UNKNOWN will be retried)
  ERROR        : 0
  UNKNOWN      : 0
pending        : 33,943 / 33,943
✅ pending workload computed


## Load prior-turn context

For each candidate we feed the LLM the **last `N_PRIOR_TURNS`
utterances** from `utterances.parquet` as conversational context,
each tagged with a per-session anonymized speaker label
(`Speaker A`, `Speaker B`, …). The current excerpt is also tagged
with its speaker so the LLM knows whose turn it's judging.

What we deliberately do **not** include:

- `parent_text` — for `speaker_change` rows it equals `text` (exact
  copy); for `intra_utterance_cut` rows the candidate `text` is a
  strict prefix. Either way it directly encodes the cut-vs-change
  distinction.
- The **next** turn — for `speaker_change` rows it's a different
  speaker on a new topic; for `intra_utterance_cut` rows it's
  literally the rest of the same sentence from the same speaker. A
  syntactic continuation would be trivially recognizable, collapsing
  this signal into a copy of the structural rule.
- `gap_to_next_s` / next-speaker identity — same leak.

Prior turns are safe: they sit *before* the boundary the LLM is
judging, so they don't tell the model whether the speaker continued
or yielded.

Speaker labels are normalized per-session: the first non-noise
speaker we see in `utt_idx` order becomes "A", the next "B", etc.
RAMC's `G00000000` non-speech bucket is dropped from the context.

In [58]:
UTTERANCES_PARQUET = MINING_ROOT / "utterances.parquet"
NON_SPEECH_SPEAKER = "G00000000"
N_PRIOR_TURNS = 3

utt = pd.read_parquet(UTTERANCES_PARQUET)

# Per-session ordered (utt_idx, speaker_id, text) triples, sorted ascending
session_utts: dict[str, list[tuple[int, str, str]]] = {}
for sess_id, group in utt.groupby("session_id"):
    g = group.sort_values("utt_idx")
    session_utts[sess_id] = list(zip(
        g["utt_idx"].astype(int).tolist(),
        g["speaker_id"].tolist(),
        g["text"].fillna("").tolist(),
    ))

# Per-session speaker_id -> 'A' / 'B' / 'C' / ... in first-seen order
# (excluding the non-speech bucket).
speaker_label_by_session: dict[str, dict[str, str]] = {}
for sess_id, items in session_utts.items():
    seen: list[str] = []
    for _, sp, _ in items:
        if sp != NON_SPEECH_SPEAKER and sp not in seen:
            seen.append(sp)
    speaker_label_by_session[sess_id] = {
        sp: chr(ord("A") + i) for i, sp in enumerate(seen)
    }


def speaker_label(session_id: str, speaker_id: str) -> str:
    return speaker_label_by_session.get(session_id, {}).get(speaker_id, "?")


def prior_context_lines(session_id: str, parent_utt_idx: int,
                        n_prior: int = N_PRIOR_TURNS) -> list[str]:
    """Return up to n_prior context lines, oldest first, formatted as
    ``[Speaker X]: text``. Skips the non-speech bucket."""
    items = session_utts.get(session_id, [])
    spk_map = speaker_label_by_session.get(session_id, {})
    prior = [
        (sp, txt)
        for idx, sp, txt in items
        if idx < parent_utt_idx and sp != NON_SPEECH_SPEAKER and txt
    ][-n_prior:]
    return [f"[Speaker {spk_map.get(sp, '?')}]: {txt}" for sp, txt in prior]


# Quick sanity: how many candidates have at least one prior line?
n_with_prior = sum(
    1
    for s, i in zip(scored["session_id"], scored["parent_utt_idx"])
    if prior_context_lines(s, int(i))
)
n_speakers_per_sess = pd.Series(
    [len(m) for m in speaker_label_by_session.values()]
).describe().round(1)

print(f"utterances loaded            : {len(utt):,}")
print(f"candidates with prior context: {n_with_prior:,} / {len(scored):,}")
print(f"speakers/session (describe)  :")
print(n_speakers_per_sess.to_string())

# Show one example so the prompt format is obvious before we run it
ex = scored.sample(1, random_state=0).iloc[0]
ex_lines = prior_context_lines(ex["session_id"], int(ex["parent_utt_idx"]))
ex_spk   = speaker_label(ex["session_id"], ex["speaker_id"])
print(f"\nexample (session {ex['session_id']}, parent_utt_idx={ex['parent_utt_idx']}):")
for line in ex_lines:
    print(f"  {line}")
print(f"  → Current excerpt by Speaker {ex_spk}: {ex['text'][:60]!r}")
print("✅ prior-turn lookup ready")

utterances loaded            : 218,408
candidates with prior context: 33,829 / 33,943
speakers/session (describe)  :
count    349.0
mean       2.0
std        0.0
min        2.0
25%        2.0
50%        2.0
75%        2.0
max        2.0

example (session CTS-CN-F2F-2019-11-15-1421, parent_utt_idx=144):
  [Speaker A]: 约六分钟，然后
  [Speaker A]: 尸体会移动并尝试离开太平间，就是非常神秘就跟
  [Speaker A]: 有一些灵魂它移不动似的
  → Current excerpt by Speaker A: '医院里边'
✅ prior-turn lookup ready


## Prompt + smoke test

The system prompt frames the task and includes 6 hand-written Chinese
few-shot examples (3 END_OF_TURN + 3 CONTINUATION) covering common
shapes: short complete reply, terminal-particle + period, trailing
conjunction, dangling subject, hesitation/filler.

The user prompt presents:

1. The last 3 turns of conversation, oldest first, each tagged
   `[Speaker X]: …` with anonymized speaker labels.
2. The current excerpt to judge, tagged with its speaker so the LLM
   knows whose turn it's evaluating.

Nothing about the **next** turn or `parent_text` reaches the model —
those would leak the structural label. See the markdown above for
details.

Verdicts are parsed by case-insensitive substring search: whichever
of `end_of_turn` / `continuation` appears first wins. Anything else
maps to `UNKNOWN` and is treated by notebook 06 as "no signal", not
"disagree".

Smoke test calls the API on 4 random rows so we catch auth /
model-id errors before the full run, and shows the actual prompt
each row produces. Each row also embeds the clip's audio
(`[clip_start_s, clip_end_s)` sliced from the session WAV without
loading the whole file) so you can listen and sanity-check whether
the verdict matches what the speaker actually does at the boundary.

In [59]:
import asyncio
import httpx
import soundfile as sf
from IPython.display import Audio, display

SYSTEM_PROMPT = (
    "You evaluate a transcribed Chinese speech excerpt and decide whether "
    "the speaker has finished their turn or paused mid-thought.\n"
    "\n"
    "- END_OF_TURN: the speaker has yielded the floor — the excerpt is a "
    "complete utterance / sentence / response.\n"
    "- CONTINUATION: the speaker paused mid-thought and intends to keep "
    "speaking — the excerpt feels truncated, mid-clause, or incomplete.\n"
    "\n"
    "Conversation context lines are shown as `[Speaker X]: text`, "
    "oldest first. Use them only as background. Judge the excerpt itself.\n"
    "\n"
    "Examples:\n"
    "\n"
    "Excerpt: 对，就是这样\n"
    "Verdict: END_OF_TURN\n"
    "\n"
    "Excerpt: 然后我就想着\n"
    "Verdict: CONTINUATION\n"
    "\n"
    "Excerpt: 我觉得这个方案挺好的。\n"
    "Verdict: END_OF_TURN\n"
    "\n"
    "Excerpt: 因为他\n"
    "Verdict: CONTINUATION\n"
    "\n"
    "Excerpt: 嗯，可以。\n"
    "Verdict: END_OF_TURN\n"
    "\n"
    "Excerpt: 就是那个，就是\n"
    "Verdict: CONTINUATION\n"
    "\n"
    "Reply with exactly one token: END_OF_TURN or CONTINUATION. "
    "No other text."
)

def build_user_prompt(text: str, prior_lines: list[str],
                      current_speaker: str) -> str:
    parts: list[str] = []
    if prior_lines:
        parts.append("Conversation so far (oldest first):")
        parts.extend(prior_lines)
        parts.append("")
    parts.append(f"Current excerpt by Speaker {current_speaker}:")
    parts.append(text)
    parts.append("")
    parts.append("Verdict:")
    return "\n".join(parts)

def parse_verdict(raw: str) -> str:
    if not raw:
        return "UNKNOWN"
    s = raw.lower()
    eot_pos = s.find("end_of_turn")
    if eot_pos == -1:
        eot_pos = s.find("end of turn")
    cont_pos = s.find("continuation")
    if eot_pos == -1 and cont_pos == -1:
        return "UNKNOWN"
    if eot_pos == -1:
        return "CONTINUATION"
    if cont_pos == -1:
        return "END_OF_TURN"
    return "END_OF_TURN" if eot_pos < cont_pos else "CONTINUATION"


class RateLimited(Exception):
    """Raised when OpenRouter returns 429. Carries the suggested wait time."""
    def __init__(self, retry_after_s: float):
        self.retry_after_s = retry_after_s
        super().__init__(f"rate limited; retry after {retry_after_s:.1f}s")


async def call_openrouter(client: httpx.AsyncClient, text: str,
                          prior_lines: list[str],
                          current_speaker: str) -> tuple[str, str]:
    """Return (verdict, raw_content). Raises RateLimited on 429,
    httpx.HTTPStatusError on other non-2xx, transport errors as-is."""
    payload = {
        "model": MODEL,
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": build_user_prompt(text, prior_lines, current_speaker)},
        ],
        "temperature": TEMPERATURE,
        "max_tokens": MAX_TOKENS,
    }
    if REASONING_EFFORT is not None:
        payload["reasoning"] = {"effort": REASONING_EFFORT}
    r = await client.post(
        "https://openrouter.ai/api/v1/chat/completions",
        headers={
            "Authorization": f"Bearer {API_KEY}",
            "HTTP-Referer": OR_REFERER,
            "X-Title": OR_TITLE,
            "Content-Type": "application/json",
        },
        json=payload,
        timeout=REQUEST_TIMEOUT_S,
    )
    if r.status_code == 429:
        try:
            wait = float(r.headers.get("Retry-After") or RATELIMIT_BACKOFF_S)
        except ValueError:
            wait = RATELIMIT_BACKOFF_S
        raise RateLimited(retry_after_s=wait)
    r.raise_for_status()
    data = r.json()
    msg = data["choices"][0]["message"]
    # Use the final answer ONLY. Reasoning text rambles and may mention
    # "continuation" / "end_of_turn" as concepts before reaching the
    # verdict — parsing it leads to wrong labels. If `content` is empty
    # (length-truncated reasoning), parse_verdict returns "UNKNOWN" and
    # the resume logic retries the row on the next run.
    raw = (msg.get("content") or "").strip()
    return parse_verdict(raw), raw


def load_clip(session_id: str, start_s: float, end_s: float):
    """Read just the [start_s, end_s) slice of a session WAV — sf.read's
    `start` / `frames` kwargs seek into the file rather than loading it
    all, so this stays cheap even though RAMC sessions are ~40 minutes."""
    path = WAV_DIR / f"{session_id}.wav"
    sr = sf.info(str(path)).samplerate
    start  = int(start_s * sr)
    frames = max(1, int((end_s - start_s) * sr))
    audio, _ = sf.read(str(path), start=start, frames=frames,
                       dtype="float32", always_2d=False)
    return audio, sr


# Smoke test on 4 rows: 2 from each label
sample = pd.concat([
    scored[scored["label"] == 1].sample(2, random_state=0),
    scored[scored["label"] == 0].sample(2, random_state=0),
]).reset_index(drop=True)

async def _smoke():
    async with httpx.AsyncClient() as client:
        for _, row in sample.iterrows():
            prior   = prior_context_lines(row["session_id"], int(row["parent_utt_idx"]))
            cur_spk = speaker_label(row["session_id"], row["speaker_id"])
            verdict, raw = await call_openrouter(client, row["text"], prior, cur_spk)
            print(f"  label={row['label']}  src={row['source']:20s}  speaker={cur_spk}  "
                  f"clip=[{row['clip_start_s']:.2f},{row['clip_end_s']:.2f}]")
            for line in prior:
                print(f"    ctx: {line[:60]}")
            print(f"    text   : {row['text'][:60]!r}")
            print(f"    verdict: {verdict}   raw[:60]: {raw[:60]!r}")
            audio, sr = load_clip(row["session_id"],
                                  float(row["clip_start_s"]),
                                  float(row["clip_end_s"]))
            display(Audio(audio, rate=sr))

await _smoke()
print("✅ smoke test complete")

  label=1  src=speaker_change        speaker=B  clip=[213.19,220.99]
    ctx: [Speaker A]: 比书本上面的知识更加的有意思，有趣味吧
    ctx: [Speaker A]: 并且以后长大，我们也可以运用到自己的生活中去，这过来人告诉我们经验
    ctx: [Speaker B]: 也可能我觉得是我们这个专业的话
    text   : '呃，对于我们电子商务老师的话，可能都是觉得比较广泛，然后也可以给我们呈现其他课外的东西'
    verdict: CONTINUATION   raw[:60]: 'CONTINUATION'


  label=1  src=speaker_change        speaker=A  clip=[1373.38,1375.97]
    ctx: [Speaker A]: 就是。
    ctx: [Speaker A]: 民族。
    ctx: [Speaker A]: 哦我真的希望那啥。
    text   : '去下那种异域风情特别厉害。'
    verdict: END_OF_TURN   raw[:60]: 'END_OF_TURN'


  label=0  src=intra_utterance_cut   speaker=B  clip=[1172.81,1176.32]
    ctx: [Speaker A]: 既然语言解决不了的问题，那就只能用行动了。
    ctx: [Speaker B]: 不能我我只我行文不行武
    ctx: [Speaker A]: 你行文不行武，但是我行武也行文
    text   : '哇，那好啊你行文你你懂得爱情吗？你'
    verdict: CONTINUATION   raw[:60]: 'CONTINUATION'


  label=0  src=intra_utterance_cut   speaker=A  clip=[1141.87,1144.38]
    ctx: [Speaker A]: 羊村开始了绝大
    ctx: [Speaker A]: 的反攻，然后与此同时嘞
    ctx: [Speaker A]: 青青草原迎来了一个不速之客来自
    text   : '就是来自月球'
    verdict: CONTINUATION   raw[:60]: 'CONTINUATION'


✅ smoke test complete


## Score every pending row

Async loop with a `Semaphore(CONCURRENCY)` cap. Each verdict is
appended to `llm_verdicts.jsonl` *as it completes* (not at the end),
so a Ctrl-C or kernel crash leaves a partial-but-resumable log on
disk. Re-running the previous cell + this one picks up exactly where
it left off — only rows without a usable verdict re-fire.

**Retry policy.**

- **429 (rate limit)** — sleep for `Retry-After` (or
  `RATELIMIT_BACKOFF_S` if absent), with jitter, then retry. Doesn't
  count against `MAX_RETRIES`. Paid Kimi rarely 429s, but the policy
  is here in case you swap to a free-tier model.
- **Other transport / HTTP errors** — exponential backoff
  (1s, 2s, 4s, …), capped at `MAX_RETRIES` attempts.

After the final retry, the row is recorded with `verdict = "ERROR"`.
On a subsequent rerun those rows are pulled back into `pending` and
get another shot. Same goes for `UNKNOWN` (the model emitted
something we couldn't parse) — they'll be retried too.

In [ ]:
import random
from tqdm.auto import tqdm

if len(pending) == 0:
    print("nothing to do — all rows already scored")
else:
    sem = asyncio.Semaphore(CONCURRENCY)
    pbar = tqdm(total=len(pending), desc="LLM verdicts")
    file_lock = asyncio.Lock()
    rl_count = 0   # observability: how many 429s have we seen?

    async def score_row(client, row, fp):
        nonlocal rl_count
        async with sem:
            verdict, raw, err = "ERROR", "", None
            attempts = 0
            while attempts < MAX_RETRIES:
                try:
                    prior   = prior_context_lines(row["session_id"], int(row["parent_utt_idx"]))
                    cur_spk = speaker_label(row["session_id"], row["speaker_id"])
                    verdict, raw = await call_openrouter(client, row["text"], prior, cur_spk)
                    err = None
                    break
                except RateLimited as rl:
                    # 429 does NOT burn an attempt — wait it out and try again.
                    rl_count += 1
                    sleep_s = rl.retry_after_s + random.uniform(0, 1)
                    pbar.set_postfix_str(f"429 backoff {sleep_s:.0f}s (total 429s={rl_count})")
                    await asyncio.sleep(sleep_s)
                except Exception as e:
                    err = repr(e)
                    attempts += 1
                    if attempts < MAX_RETRIES:
                        await asyncio.sleep(2 ** (attempts - 1))
            rec = {
                "session_id":    row["session_id"],
                "candidate_idx": int(row["candidate_idx"]),
                "verdict":       verdict,
                "raw":           raw,
                "model":         MODEL,
            }
            if err is not None:
                rec["error"] = err
            line = json.dumps(rec, ensure_ascii=False) + "\n"
            async with file_lock:
                fp.write(line)
                fp.flush()
            pbar.update(1)

    async def _run():
        limits = httpx.Limits(max_connections=CONCURRENCY * 2,
                              max_keepalive_connections=CONCURRENCY)
        async with httpx.AsyncClient(limits=limits) as client:
            with LLM_JSONL.open("a", encoding="utf-8") as fp:
                tasks = [score_row(client, row, fp) for _, row in pending.iterrows()]
                await asyncio.gather(*tasks)

    await _run()
    pbar.close()
    print(f"scoring loop finished — total 429s seen: {rl_count}")
    print("✅ scoring loop finished")

## Merge verdicts back into a parquet table

Read every line of `llm_verdicts.jsonl`, dedupe on
`(session_id, candidate_idx)` keeping the **last** record (so reruns
override stale verdicts), join onto `candidates_scored.parquet`, and
write `candidates_llm.parquet`.

In [ ]:
records: dict[tuple[str, int], dict] = {}
with LLM_JSONL.open("r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        try:
            rec = json.loads(line)
        except json.JSONDecodeError:
            continue
        key = (rec["session_id"], int(rec["candidate_idx"]))
        records[key] = rec   # last write wins

verdicts_df = pd.DataFrame(
    [
        {
            "session_id":    k[0],
            "candidate_idx": k[1],
            "llm_verdict":   v["verdict"],
            "llm_raw":       v.get("raw", ""),
            "llm_model":     v.get("model", ""),
        }
        for k, v in records.items()
    ]
)

merged = scored.merge(verdicts_df, on=["session_id", "candidate_idx"], how="left")
merged["llm_verdict"] = merged["llm_verdict"].fillna("MISSING")

tmp = LLM_PARQUET.with_suffix(".parquet.tmp")
merged.to_parquet(tmp, index=False)
tmp.replace(LLM_PARQUET)

print(f"rows         : {len(merged):,}")
print(f"verdict dist :")
print(merged["llm_verdict"].value_counts().to_string())
print(f"out          : {LLM_PARQUET.name} ({LLM_PARQUET.stat().st_size / 1e6:.1f} MB)")
print("✅ candidates_llm.parquet written")

## Calibration view

Two views over the LLM verdicts:

1. **Agreement with the structural label** — how often does the LLM's
   verdict match the rule-based label, broken out by source.
2. **Pairwise agreement matrix (structural × model × LLM)** — counts
   per `(label, model_pred, llm)` triple. Rows where all three agree
   are notebook 06's auto-accept candidates; rows where they disagree
   route to human review.

Diagnostic only — the consensus router (notebook 06) decides
acceptance, not this cell.

In [ ]:
ok = merged[merged["llm_verdict"].isin(["END_OF_TURN", "CONTINUATION"])].copy()
ok["llm_label"] = (ok["llm_verdict"] == "END_OF_TURN").astype("int8")

print(f"usable verdicts : {len(ok):,} / {len(merged):,}")
print(f"unknown / error / missing : {len(merged) - len(ok):,}")

print("\nLLM agreement with structural label:")
for source, sub in ok.groupby("source"):
    agree = (sub["llm_label"] == sub["label"]).mean()
    print(f"  {source:22s} {len(sub):>6,}  agree={agree:.3f}")

print("\nLLM agreement with own-model prediction (rows where model_pred ∈ {0,1}):")
ok_m = ok[ok["model_pred"].isin([0, 1])]
agree = (ok_m["llm_label"] == ok_m["model_pred"]).mean() if len(ok_m) else float("nan")
print(f"  overall            {len(ok_m):>6,}  agree={agree:.3f}")

print("\ntri-signal agreement counts (label, model_pred, llm_label):")
tri = (
    ok_m
    .groupby(["label", "model_pred", "llm_label"])
    .size()
    .reset_index(name="n")
    .sort_values("n", ascending=False)
)
print(tri.to_string(index=False))

unanimous_yes = ((ok_m["label"] == 1) & (ok_m["model_pred"] == 1) & (ok_m["llm_label"] == 1)).sum()
unanimous_no  = ((ok_m["label"] == 0) & (ok_m["model_pred"] == 0) & (ok_m["llm_label"] == 0)).sum()
print(f"\nunanimous end-of-turn : {unanimous_yes:,}")
print(f"unanimous continuation : {unanimous_no:,}")
print(f"=> {unanimous_yes + unanimous_no:,} rows ready for auto-accept (subject to notebook 06 routing)")
print("✅ calibration printed")